# Download, Host, and Query Your Own AI Model

## Classroom learning goals 

By the end of this notebook, you will be able to:

- Explain the difference between downloading a model and loading it into memory.
- Identify the tokenizer and the model weights, and describe what each part does.
- Check whether the model will run on a CPU or a GPU.
- Send prompts to a locally hosted model and inspect its responses.
- Describe the limits of a base language model.

## What will happen?

1. **Install dependencies:** Add Python libraries that can download and run the model.
2. **Choose storage:** Create a local folder where downloaded model files can be cached.
3. **Check hardware:** Determine whether a GPU is available; otherwise, use the CPU.
4. **Download and load:** Retrieve the model and tokenizer, then move the model into memory.
5. **Test the model:** Ask several prepared questions and measure generation time.
6. **Explore interactively:** Enter your own prompts in a text box.

> **Important:** In this notebook, “hosting” means running the model inside the current Jupyter process. It is not a public web service or API.

---

In [ ]:
# STEP 1: Install the Python libraries used by this notebook.
#
# - transformers: downloads and runs the model and tokenizer
# - torch: provides the numerical runtime used for AI calculations
# - accelerate: helps Transformers manage model loading
# - ipywidgets: creates the interactive text box and button
#
# %pip uses the Jupyter kernel's Python environment, unlike a plain system pip command.
%pip install --quiet transformers ipywidgets torch accelerate

print("Required libraries are installed.")

## Step 2: Configure Local Model Storage

The next cell creates an `ai_models` cache folder in the current user's home directory.

**Why cache downloaded files?**
- The first download may take several minutes.
- Later sessions can reuse downloaded files instead of repeating the transfer.
- A visible local cache helps explain the difference between downloading and loading.

The code creates the directory if it does not already exist.

In [ ]:
# STEP 2: Create a folder for cached model files.
from pathlib import Path

# The cache belongs to the current user and is independent of the notebook directory.
MODELS_DIR = Path.home() / "ai_models"

# mkdir(..., exist_ok=True) creates the folder if needed and does not fail if it exists.
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model cache directory: {MODELS_DIR}")
print("This folder may contain downloaded tokenizer files and model weights.")

## Step 3: Choose and Load a Language Model

A language model needs computational resources to generate text. GPU execution is faster, but the FLAN-T5 options can run on a CPU.

### Model options

- **Beginner:** `google/flan-t5-small` — smaller and fastest.
- **Beginner:** `google/flan-t5-base` — currently selected; better quality than small.
- **Intermediate:** `Qwen/Qwen2.5-1.5B-Instruct` — conversational/instruction model.
- **Advanced:** `Qwen/Qwen2.5-7B-Instruct` — larger and usually requires a GPU with enough memory.

### FLAN-T5 versus Qwen configuration

**FLAN-T5** is a sequence-to-sequence model:
- Load with `AutoModelForSeq2SeqLM`.
- Send a plain-text prompt.
- Beam search is suitable for these demonstrations.

**Qwen** is a causal/instruction model:
- Load with `AutoModelForCausalLM`.
- Format prompts with `apply_chat_template()` and a `user` role.
- Set `torch_dtype="auto"` for appropriate precision.
- The 7B model may require 4-bit loading with `bitsandbytes`.

Only one model should be selected. In the loading cell, uncomment the preferred model block and comment out every other selection.

In [ ]:
# STEP 3A: Detect available hardware.
#
# CUDA indicates an NVIDIA GPU is available through PyTorch.
# If CUDA is unavailable, the notebook uses the CPU instead.
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
print("Selected device:", DEVICE.upper())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print("GPU:", gpu_name)
    print("GPU memory:", round(gpu_memory_gb, 1), "GB")
else:
    print("The CPU will be used. Generation may be slower.")

In [ ]:
# STEP 3B: Select exactly one model, then uncomment/loading matching the Qwen family block.
from transformers import AutoTokenizer

# BEGINNER — currently selected
MODEL_FAMILY = "flan"
model_name = "google/flan-t5-base"

# BEGINNER — uncomment this alternative and comment out the three active lines above.
# MODEL_FAMILY = "flan"
# model_name = "google/flan-t5-small"

# INTERMEDIATE — uncomment this alternative and comment out the active FLAN lines above.
# MODEL_FAMILY = "qwen"
# model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# ADVANCED — uncomment this alternative and comment out the active FLAN lines above.
# MODEL_FAMILY = "qwen"
# model_name = "Qwen/Qwen2.5-7B-Instruct"

print(f"Model: {model_name}")
print(f"Model family: {MODEL_FAMILY}")
print(f"Cache directory: {MODELS_DIR}")

# --------------------------------------------------
# FLAN-T5 MODELS: active sequence-to-sequence loading
# --------------------------------------------------
# Keep this block active only when MODEL_FAMILY is "flan".
from transformers import AutoModelForSeq2SeqLM

print("Loading FLAN-T5 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=MODELS_DIR,
)

print("Loading FLAN-T5 model weights...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    cache_dir=MODELS_DIR,
).to(DEVICE)

# --------------------------------------------------
# QWEN MODELS: alternative causal/instruction loading
# --------------------------------------------------
# Select one Qwen model above and replace the active FLAN-T5 block above with:
#
# from transformers import AutoModelForCausalLM

# print("Loading Qwen tokenizer...")
# tokenizer = AutoTokenizer.from_pretrained(
#     model_name,
#     cache_dir=MODELS_DIR,
#     trust_remote_code=True,
# )

# print("Loading Qwen model weights...")
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     cache_dir=MODELS_DIR,
#     torch_dtype="auto",
#     trust_remote_code=True,
# ).to(DEVICE)

# QWEN 7B / MEMORY-LIMITED GPU OPTION:
# 1. Install 4-bit support: %pip install --quiet bitsandbytes
# 2. Comment out the normal .to(DEVICE) loading block above.
# 3. Uncomment the alternative block below.

# from transformers import BitsAndBytesConfig
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     cache_dir=MODELS_DIR,
#     quantization_config=quantization_config,
#     device_map="auto",
#     trust_remote_code=True,
# )

print("The model and tokenizer are hosted by this notebook process.")
print("The model is ready to receive prompts.")

## Query the hosted model

In [ ]:
questions = [
    "What are the main components of the Solar System?",
    "How does the Internet work at a high level?",
    "What are some applications of artificial intelligence?",
    "Explain what makes a large language model useful.",
]

In [ ]:
# STEP 4B: Define prompt formatting and generation for the selected model family.
def ask(model, tokenizer, question, max_new_tokens=200):
    """Send a prompt directly to the selected locally hosted model.

    FLAN-T5: plain text prompt, sequence-to-sequence generation.
    Qwen: chat-template prompt, causal/instruction generation.
    """
    if MODEL_FAMILY == "flan":
        # FLAN-T5 path — active for flan-t5-small and flan-t5-base.
        inputs = tokenizer(question, return_tensors="pt").to(DEVICE)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            no_repeat_ngram_size=2,
        )
        answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    elif MODEL_FAMILY == "qwen":
        # QWEN path — use this for Qwen2.5-1.5B-Instruct or Qwen2.5-7B-Instruct.
        # Apply the Qwen chat template and mark the prompt as ready for an answer.
        messages = [{"role": "user", "content": question}]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

        # Remove the prompt tokens so only the model's new answer is displayed.
        answer = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )

    else:
        raise ValueError("MODEL_FAMILY must be 'flan' or 'qwen'.")

    print("Model:", model_name)
    print("Q:", question)
    print("A:", answer)
    print("-" * 80)
    return answer


# --------------------------------------------------
# FLAN-T5 ONLY: fully commented example of the older function
# --------------------------------------------------
# def ask(model, tokenizer, question, max_new_tokens=200):
#     inputs = tokenizer(question, return_tensors="pt").to(DEVICE)
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         num_beams=4,
#         no_repeat_ngram_size=2,
#     )
#     answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     print("Q:", question)
#     print("A:", answer)
#     print("-" * 80)
#     return answer

# --------------------------------------------------
# QWEN ONLY: fully commented example of the chat function
# --------------------------------------------------
# def ask(model, tokenizer, question, max_new_tokens=200):
#     messages = [{"role": "user", "content": question}]
#     prompt = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         do_sample=True,
#         temperature=0.7,
#         top_p=0.9,
#     )
#     answer = tokenizer.decode(
#         outputs[0][inputs["input_ids"].shape[1]:],
#         skip_special_tokens=True,
#     )
#     print("Q:", question)
#     print("A:", answer)
#     print("-" * 80)
#     return answer

## Run sample questions

The questions below let us verify that the downloaded model is loaded and generating answers. Total runtime is shown at the end.

In [ ]:
%%time

for question in questions:
    ask(model, tokenizer, question)

## Step 6: Use the Interactive Prompt

This final interface lets each student type an original question and receive a response from the model running in this notebook.

### How the interface works

- `Text` creates the question input box.
- `Button` creates the button that submits the prompt.
- `Output` is a dedicated area where the response appears.
- `submit_question()` reads the input, clears the previous response, and calls `ask()`.
- Pressing **Enter** or clicking **Ask model** starts generation.

### Classroom activities

1. Ask one straightforward factual question.
2. Ask a follow-up with a different wording.
3. Ask the model to explain a concept at a beginner level.
4. Compare the response with a known source.
5. Ask an ambiguous question and discuss how the model handles missing context.

> **Model limitation:** This notebook uses a base model from the Transformers library, not a live Internet-connected assistant. It cannot browse the web or verify facts unless they were learned during model training.

In [ ]:
# Interactive prompt
# Enter a question in the text box and the locally hosted model will respond.
import ipywidgets as widgets
from IPython.display import display

question_box = widgets.Text(
    placeholder="Ask the local model a question...",
    description="Question:",
    layout=widgets.Layout(width="80%"),
)
ask_button = widgets.Button(description="Ask model", button_style="primary")
output = widgets.Output()

def submit_question(_):
    question = question_box.value.strip()
    if not question:
        return

    with output:
        output.clear_output(wait=True)
        ask(model, tokenizer, question, max_new_tokens=300)

ask_button.on_click(submit_question)
# question_box.on_submit(submit_question)

display(question_box, ask_button, output)

print("Tip: press Enter or click 'Ask model' to submit a question.")